# 孤独・孤立実態調査 分析レポート（令和6年度）

**出典**: 孤独・孤立対策に関する実態調査（内閣官房 孤独孤立対策担当室、令和6年度）
**対象**: 全国の一般市民（16歳以上）
**有効回答数**: 10,871人（孤独感ファイル）
**データ**: `data03/01.孤独感に関する集計表/` 令和6年度版 (r6)20250423
**分析**: UCLA孤独感尺度・直接質問・社会的交流・社会参加・行政支援の5テーマ

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント
plt.rcParams['font.family'] = ['Yu Gothic', 'Meiryo', 'MS Gothic', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# パス設定
BASE = Path('.')
D = BASE / 'data03' / '01.孤独感に関する集計表'
IMG_DIR = BASE / 'レポート' / 'images'
IMG_DIR.mkdir(parents=True, exist_ok=True)

F01 = D / '01 孤独感に関する集計表(r6)20250423.xlsx'
F02 = D / '02 孤立（社会的交流）に関する集計表(r6)20250423.xlsx'
F03 = D / '03 孤立（社会参加）に関する集計表(r6)20250423.xlsx'
F04 = D / '04 孤立（各種支援）に関する集計表(r6)20250423.xlsx'

print('セットアップ完了')
print(f'データフォルダ: {D}')

## Section 1: データ読み込みユーティリティ

In [ ]:
def read_pct(filepath, sheet_name, n_label_cols=1):
    """
    survey Excel クロス集計シートからパーセンテージ行を読み込む。

    Parameters
    ----------
    filepath : str or Path
    sheet_name : str
    n_label_cols : int
        n/100列の左側にあるカテゴリラベル列の数
        - 都市規模別: 1
        - 性×年齢: 2
        - 性×就業状況 等: 3
        - 孤独感×社会参加等: 4

    Returns
    -------
    pd.DataFrame
        index: ラベル（重複あり → iloc で位置指定して使用）
        columns: 設問回答選択肢
        values: パーセンテージ値

    Notes
    -----
    - pct行の判定: n列の値が 100.0
    - count行: n列の値が実人数
    - 無回答・欠損は np.nan
    """
    raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None)
    hrow = None
    for i in range(len(raw)):
        for j in range(len(raw.columns)):
            v = raw.iloc[i, j]
            if pd.notna(v) and '回答者数' in str(v):
                hrow = i; break
        if hrow is not None: break
    if hrow is None:
        raise ValueError(f'回答者数行が見つかりません: {sheet_name}')

    n_col = n_label_cols
    data_start = n_label_cols + 1
    cols_raw = raw.iloc[hrow, data_start:].tolist()
    cols = [str(c).strip().replace('\n', ' ') if pd.notna(c) else '' for c in cols_raw]

    rows, labels = [], []
    cur_label = [''] * n_label_cols
    for i in range(hrow + 1, len(raw)):
        row = raw.iloc[i]
        for j in range(n_label_cols):
            v = row.iloc[j]
            if pd.notna(v) and str(v).strip():
                cur_label[j] = str(v).strip()
        nv = row.iloc[n_col]
        if pd.notna(nv):
            try: nv_f = float(nv)
            except: continue
            if nv_f == 100.0:
                pct = []
                for c_idx in range(data_start, data_start + len(cols)):
                    if c_idx < len(row):
                        v = row.iloc[c_idx]
                        if pd.isna(v) or str(v).strip() in ('-', ''): pct.append(np.nan)
                        else:
                            try: pct.append(float(v))
                            except: pct.append(np.nan)
                    else: pct.append(np.nan)
                label = next((l for l in reversed(cur_label) if l), '不明')
                rows.append(pct[:len(cols)])
                labels.append(label)

    if not rows: return pd.DataFrame()
    return pd.DataFrame(rows, index=labels, columns=cols[:len(rows[0])])

def save_fig(fig, name):
    """画像を images/ に保存"""
    path = IMG_DIR / name
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor='white')
    print(f'  → {path.name} 保存')

print('ユーティリティ関数定義完了')

## Section 2: 孤独感の実態

### 2.1 孤独感測定方法
本調査では2種類の孤独感測定方法を併用している：

| 測定法 | 内容 | 選択肢 |
|--------|------|--------|
| **UCLA孤独感尺度** | 3問の合計スコア（3〜12点） | 10-12点（常にある）/ 7-9点（時々ある）/ 4-6点（ほとんどない）/ 3点（決してない） |
| **直接質問** | 孤独を感じるか | しばしばある・常にある / 時々ある / たまにある / ほとんどない / 決してない |

In [ ]:
# ── 2.2 UCLA孤独感尺度 都市規模別分布 ──
df_11 = read_pct(F01, '1-1', n_label_cols=1)
cities_short = ['全体', '政令市・特別区', '20万人以上市', '10-20万人市', '10万人未満市', '町村']
data_11 = df_11.iloc[0:6, 0:4].fillna(0).copy()
data_11.index = cities_short

print('=== 都市規模別 UCLA孤独感スコア ===')
print(data_11.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#c0392b', '#e67e22', '#3498db', '#27ae60']
labels_legend = ['10-12点（常にある）', '7-9点（時々ある）', '4-6点（ほとんどない）', '3点（決してない）']
left = np.zeros(len(data_11))
for i, (color, label) in enumerate(zip(colors, labels_legend)):
    vals = data_11.iloc[:, i].values
    ax.barh(data_11.index, vals, left=left, color=color, label=label, edgecolor='white', linewidth=0.5)
    for j, (v, l) in enumerate(zip(vals, left)):
        if v > 3:
            ax.text(l + v/2, j, f'{v:.1f}%', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    left += vals
ax.set_xlabel('割合（%）', fontsize=11)
ax.set_title('UCLA孤独感尺度 都市規模別分布（令和6年度調査 n=10,871）', fontsize=13, fontweight='bold', pad=12)
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax.set_xlim(0, 105); ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
save_fig(fig, 'fig01.png')
plt.show()

**【考察 2.2】**
- 全体では UCLA高孤独（10-12点）が **6.5%**、孤独「時々ある」（7-9点）が **39.2%** で、合計 **45.7%** が何らかの孤独感を抱えている
- 都市規模別の差は小さく（5.8〜7.4%）、孤独感は特定の地域に偏らない全国的な問題
- 政令市・特別区でやや高め（7.4%）→ 都市部の匿名性・繋がりの希薄化が示唆される

In [ ]:
# ── 2.3 年齢別 孤独感の変化 ──
df_12 = read_pct(F01, '1-2', n_label_cols=2)
data_12_age = df_12.iloc[1:8].fillna(0)  # 16-19歳〜70代（性別計）
age_labels = ['16-19歳', '20代', '30代', '40代', '50代', '60代', '70代']
ucla_high = data_12_age.iloc[:, 0].values
direct_high = data_12_age.iloc[:, 5].values

print('=== 年齢別 孤独感の高さ ===')
for age, u, d in zip(age_labels, ucla_high, direct_high):
    print(f'  {age}: UCLA高スコア {u:.1f}%  直接質問「しばしばある」{d:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(age_labels))
ax.plot(x, ucla_high, marker='o', color='#c0392b', linewidth=2.5, markersize=8, label='UCLA高孤独（10-12点）')
ax.plot(x, direct_high, marker='s', color='#e67e22', linewidth=2.5, markersize=8, label='直接質問「しばしばある・常にある」')
for i, (u, d) in enumerate(zip(ucla_high, direct_high)):
    ax.annotate(f'{u:.1f}%', (x[i], u), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9, color='#c0392b')
    ax.annotate(f'{d:.1f}%', (x[i], d), textcoords='offset points', xytext=(0, -16), ha='center', fontsize=9, color='#e67e22')
ax.set_xticks(x); ax.set_xticklabels(age_labels, fontsize=11)
ax.set_ylabel('割合（%）', fontsize=11)
ax.set_title('年齢別 孤独感の高さ（令和6年度調査）', fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10, loc='upper right')
ax.set_ylim(0, max(max(ucla_high), max(direct_high)) * 1.5)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
save_fig(fig, 'fig02.png')
plt.show()

**【考察 2.3】**
- **20〜30代が最も孤独感が高い**：UCLA高スコア 20代9.1%・30代9.5%（全体平均6.5%の約1.5倍）
- **高齢層では低下**：70代のUCLA高スコアは 3.9%（全体比▲2.6pt）
- 直接質問では 20代の「しばしばある・常にある」が 7.4% と最高水準
- 若年層の孤独感の高さは、就職・転居・人間関係の断絶等が背景と推測される

In [ ]:
# ── 2.4 世帯構成別 孤独感 ──
df_15 = read_pct(F01, '1-5', n_label_cols=3)
# iloc[0]=全体, [1]=ひとり世帯〜[6]=その他
household_labels = ['全体', 'ひとり世帯', '夫婦のみ', '両親と子（二世代）', 'ひとり親と子', '三世代世帯', 'その他の世帯']
data_15 = df_15.iloc[0:7, 0:4].fillna(0)
data_15.index = household_labels

print('=== 世帯構成別 UCLA高スコア率（10-12点） ===')
for idx, val in zip(household_labels, data_15.iloc[:, 0]):
    print(f'  {idx}: {val:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
colors = ['#c0392b', '#e67e22', '#3498db', '#27ae60']
labels_legend = ['10-12点（常にある）', '7-9点（時々ある）', '4-6点（ほとんどない）', '3点（決してない）']
left = np.zeros(len(data_15))
for i, (color, label) in enumerate(zip(colors, labels_legend)):
    vals = data_15.iloc[:, i].values
    ax.barh(data_15.index, vals, left=left, color=color, label=label, edgecolor='white', linewidth=0.5)
    for j, (v, l) in enumerate(zip(vals, left)):
        if v > 3:
            ax.text(l + v/2, j, f'{v:.1f}%', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    left += vals
ax.set_xlabel('割合（%）', fontsize=11)
ax.set_title('世帯構成別 孤独感の分布（UCLA孤独感尺度）', fontsize=13, fontweight='bold', pad=12)
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(0, 105); ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
save_fig(fig, 'fig03.png')
plt.show()

**【考察 2.4】**
- **ひとり世帯が最も孤独感高い**：UCLA高スコア 9.9%（全体比+3.4pt）
- **夫婦のみ世帯が最も低い**：5.0%（安定した親密な関係の存在が緩衝機能）
- **ひとり親と子世帯**は 8.0%と高水準 → 経済的・時間的プレッシャーが背景に
- 三世代世帯（5.8%）は比較的低め → 多世代のつながりが孤独感を和らげる可能性

In [ ]:
# ── 2.5 就業状況別 孤独感 ──
df_17 = read_pct(F01, '1-7', n_label_cols=3)
# iloc[0]=全体, [1]-[9]=各就業状況
emp_labels = ['全体', '正社員', '非正規', '会社役員', '自営業', '家族従業・内職', '学生', '求職中', '非求職（仕事探していない）', 'その他']
data_17 = df_17.iloc[0:10].fillna(0)
data_17.index = emp_labels
avg_val = float(df_17.iloc[0, 0])

print('=== 就業状況別 UCLA高スコア率 ===')
for idx, val in zip(emp_labels, data_17.iloc[:, 0]):
    diff = val - avg_val
    marker = '★' if abs(diff) > 3 else ' '
    print(f'  {marker}{idx}: {val:.1f}% (全体比 {diff:+.1f}pt)')

In [ ]:
ucla_vals4 = data_17.iloc[:, 0].values
fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = ['#2e86de' if i == 0 else ('#c0392b' if v >= avg_val * 1.5 else '#e67e22' if v >= avg_val else '#3498db') for i, v in enumerate(ucla_vals4)]
bars = ax.barh(emp_labels, ucla_vals4, color=colors_bar, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, ucla_vals4):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')
ax.axvline(avg_val, color='gray', linestyle='--', alpha=0.7, label=f'全体平均 {avg_val:.1f}%')
ax.set_xlabel('UCLA高孤独（10-12点）割合（%）', fontsize=11)
ax.set_title('就業状況別 UCLA高孤独スコア率（令和6年度）', fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10)
ax.invert_yaxis()
ax.set_xlim(0, max(ucla_vals4) * 1.3)
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
save_fig(fig, 'fig04.png')
plt.show()

**【考察 2.5】**
- **求職中（仕事を探している）が突出して高い**：18.6%（全体比+12.1pt）→ 就業不安定・経済的不安が大きく影響
- **正社員・非正規**は全体並み（6-7%台）
- **会社役員**は最低（0.9%）→ 社会的地位・経済的安定が孤独感の緩衝となる可能性
- **学生**（3.9%）と**自営業**（4.2%）はやや低め

In [ ]:
# ── 2.6 社会参加有無別 孤独感の比較 ──
df_117 = read_pct(F01, '1-17', n_label_cols=3)
# iloc[1]=参加あり（年齢計）, [2]=参加なし（年齢計）
data_117 = df_117.iloc[[1, 2]].fillna(0)
data_117.index = ['社会活動に参加している', '特に参加していない']
print('=== 社会参加有無別 孤独感（年齢計） ===')
print(f"  参加あり: UCLA高スコア {data_117.iloc[0, 0]:.1f}%,  直接質問「しばしばある」{data_117.iloc[0, 5]:.1f}%")
print(f"  参加なし: UCLA高スコア {data_117.iloc[1, 0]:.1f}%,  直接質問「しばしばある」{data_117.iloc[1, 5]:.1f}%")
ucla_diff = data_117.iloc[1, 0] - data_117.iloc[0, 0]
direct_diff = data_117.iloc[1, 5] - data_117.iloc[0, 5]
print(f"  差（参加なし - 参加あり）: UCLA {ucla_diff:+.1f}pt, 直接質問 {direct_diff:+.1f}pt")

In [ ]:
ucla_vals5 = data_117.iloc[:, 0].values
direct_vals5 = data_117.iloc[:, 5].values
x = np.arange(2); width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, ucla_vals5, width, color=['#2980b9', '#c0392b'], label='UCLA高孤独（10-12点）', alpha=0.9, edgecolor='white')
bars2 = ax.bar(x + width/2, direct_vals5, width, color=['#27ae60', '#e74c3c'], label='直接質問「しばしばある・常にある」', alpha=0.85, hatch='//', edgecolor='white')
for bar, val in zip(bars1, ucla_vals5):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3, f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')
for bar, val in zip(bars2, direct_vals5):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3, f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(data_117.index, fontsize=12)
ax.set_ylabel('割合（%）', fontsize=11)
ax.set_title('社会参加有無別 孤独感の比較（令和6年度調査）', fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10)
ax.set_ylim(0, max(max(ucla_vals5), max(direct_vals5)) * 1.4)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
save_fig(fig, 'fig05.png')
plt.show()

**【考察 2.6】**
- **社会参加なし群のUCLA高スコア 9.5% vs 参加あり群 3.4%**（差：+6.1pt）
- 直接質問でも参加なし 5.6% vs 参加あり 2.9%（差：+2.7pt）
- 因果関係の方向は不明だが、社会活動への参加が孤独感の緩衝になる/または孤独な人が参加しにくい、のどちらかまたは両方と考えられる

## Section 3: 社会的交流（コミュニケーション頻度）

In [ ]:
# ── 3.1 コミュニケーション頻度 全体分布 ──
df_21 = read_pct(F02, '2-1', n_label_cols=1)
total_row = df_21.iloc[0].fillna(0)
freq_labels = ['週4-5回以上', '週2-3回', '週1回', '2週に1回', '月1回', '月1回未満', '全くない', '無回答']
method_data = {
    '①直接会って話す': total_row.iloc[0:8].values,
    '②電話（ビデオ通話含む）': total_row.iloc[8:16].values,
    '③SNS・電子メール': total_row.iloc[16:24].values,
}
print('=== コミュニケーション頻度 全体 ===')
for method, vals in method_data.items():
    print(f'\n{method}')
    for fl, v in zip(freq_labels, vals):
        print(f'  {fl}: {v:.1f}%')

In [ ]:
colors_freq = ['#2e86de','#54a0ff','#6c8ebf','#82b0ff','#aecbfa','#d1e8ff','#e74c3c','#bdc3c7']
fig, axes = plt.subplots(3, 1, figsize=(12, 7))
for ax, (method, vals) in zip(axes, method_data.items()):
    left = 0
    for j, (val, fl, col) in enumerate(zip(vals, freq_labels, colors_freq)):
        ax.barh([0], [val], left=left, color=col, edgecolor='white', height=0.6)
        if val > 4:
            ax.text(left + val/2, 0, f'{val:.1f}%', ha='center', va='center',
                    fontsize=9.5, color='white' if j < 6 else 'black', fontweight='bold')
        left += val
    ax.set_xlim(0, 105)
    ax.set_yticks([0]); ax.set_yticklabels([method], fontsize=11); ax.set_ylim(-0.5, 0.5)
    ax.grid(axis='x', alpha=0.3)
axes[0].set_title('コミュニケーション手段別 頻度分布（令和6年度調査 全体）', fontsize=13, fontweight='bold', pad=12)
axes[2].set_xlabel('割合（%）', fontsize=11)
handles = [mpatches.Patch(color=c, label=l) for c, l in zip(colors_freq, freq_labels)]
fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.06))
fig.tight_layout()
save_fig(fig, 'fig06.png')
plt.show()

**【考察 3.1】**
- **③SNS・電子メールが最もよく使われる**：週4-5回以上が 20.9%（最高頻度）
- **①直接会って話す**：「全くない」が **9.3%** と最も高い（孤立リスクの指標）
- **②電話**：「全くない」14.9%、無回答 13.0%と多い → 特に高齢層で電話コミュニケーションが減少傾向
- 「無回答」が①15.5%・②13.0%と多い → 同居家族のいない単身世帯での非回答も含む

In [ ]:
# ── 3.2 年齢別コミュニケーション「全くない」率 ──
df_22 = read_pct(F02, '2-2', n_label_cols=2)
data_22_age = df_22.iloc[1:8].fillna(0)  # 16-19歳〜70代
age_labels7 = ['16-19歳', '20代', '30代', '40代', '50代', '60代', '70代']
method1_none = data_22_age.iloc[:, 6].values   # ①全くない
method2_none = data_22_age.iloc[:, 14].values  # ②全くない
method3_none = data_22_age.iloc[:, 22].values  # ③全くない

print('=== 年齢別「全くない」率 ===')
print(f'  {'年齢':10s}  直接会う  電話    SNS')
for age, v1, v2, v3 in zip(age_labels7, method1_none, method2_none, method3_none):
    print(f'  {age:10s}  {v1:5.1f}%  {v2:5.1f}%  {v3:5.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(age_labels7))
ax.plot(x, method1_none, marker='o', color='#2e86de', linewidth=2.5, markersize=8, label='①直接会って話す（全くない）')
ax.plot(x, method2_none, marker='s', color='#e67e22', linewidth=2.5, markersize=8, label='②電話（全くない）')
ax.plot(x, method3_none, marker='^', color='#27ae60', linewidth=2.5, markersize=8, label='③SNS・電子メール（全くない）')
for i in range(len(age_labels7)):
    ax.annotate(f'{method1_none[i]:.0f}%', (x[i], method1_none[i]), xytext=(0, 8),
                textcoords='offset points', ha='center', fontsize=8.5, color='#2e86de')
ax.set_xticks(x); ax.set_xticklabels(age_labels7, fontsize=11)
ax.set_ylabel('「全くない」割合（%）', fontsize=11)
ax.set_title('年齢別 コミュニケーション「全くない」率（令和6年度調査）', fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10, loc='upper left')
ax.set_ylim(0, max(method1_none.max(), method2_none.max(), method3_none.max()) * 1.4)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
save_fig(fig, 'fig07.png')
plt.show()

**【考察 3.2】**
- **16-19歳は直接会っての交流が極端に高頻度**（週4-5回以上 71.9%）→ 学校での対面交流が主
- **直接会っての「全くない」率は高齢層で上昇**：70代 16.5%、16-19歳は 4.6%
- **SNS利用「全くない」率**は若年層で極めて低く（16-19歳 5.6%）、高齢層で急増（70代 30.3%相当）
- 電話での「全くない」率は中年層（40-50代）で高い → 仕事中心の生活スタイルを反映

## Section 4: 社会参加の実態

In [ ]:
# ── 4.1 社会参加活動種別 全体 ──
df_31 = read_pct(F03, '3-1', n_label_cols=1)
total_row8 = df_31.iloc[0].fillna(0)
act_labels = ['いずれかの
活動に参加', 'PTA・自治会
町内会', '家族外の人
への手助け', 'ボランティア
活動', 'スポーツ
趣味娯楽', 'その他の
活動', '特に参加は
していない']
act_vals = total_row8.iloc[0:7].values

print('=== 社会参加活動種別 全体 ===')
for label, val in zip(act_labels, act_vals):
    print(f'  {label.replace(chr(10), " ")}: {val:.1f}%')
print(f'\n  無回答: {float(total_row8.iloc[7]):.1f}%')

In [ ]:
act_colors = ['#2e86de','#54a0ff','#5f27cd','#6c5ce7','#00b894','#fdcb6e','#e74c3c']
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(act_labels, act_vals, color=act_colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, act_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.7, f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('割合（%）', fontsize=11)
ax.set_title('社会活動への参加状況（複数回答、令和6年度調査 n=10,871）', fontsize=13, fontweight='bold', pad=12)
ax.set_ylim(0, max(act_vals) * 1.18)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
save_fig(fig, 'fig08.png')
plt.show()

**【考察 4.1】**
- **いずれかの活動に参加 46.6%**（約半数が何らかの社会活動）
- 最多は**スポーツ・趣味・娯楽（32.0%）** → 個人的な趣味活動が中心
- 地域コミュニティ活動（PTA・自治会）17.3% → 参加率は低くない
- **特に参加なし 50.6%** → 約半数は何の社会活動も行っていない

In [ ]:
# ── 4.2 孤独感レベル別 社会参加率（クロス分析） ──
df_317 = read_pct(F03, '3-17', n_label_cols=4)
# iloc[0]=全体, [1-4]=UCLA区分, [5-9]=直接質問区分
ucla_idxs = [1, 2, 3, 4]
direct_idxs = [5, 6, 7, 8, 9]
ucla_labels_x = ['10-12点\n（常にある）', '7-9点\n（時々ある）', '4-6点\n（ほぼない）', '3点\n（決してない）']
direct_labels_x = ['しばしばある\n・常にある', '時々ある', 'たまにある', 'ほとんど\nない', '決して\nない']
ucl_part = [float(df_317.iloc[i, 0]) if pd.notna(df_317.iloc[i, 0]) else 0 for i in ucla_idxs]
dir_part = [float(df_317.iloc[i, 0]) if pd.notna(df_317.iloc[i, 0]) else 0 for i in direct_idxs]

print('=== 孤独感レベル別 社会参加率 ===')
print('UCLA尺度別:')
for label, val in zip(['10-12点', '7-9点', '4-6点', '3点'], ucl_part):
    print(f'  {label}: {val:.1f}%')
print('直接質問別:')
for label, val in zip(['しばしばある', '時々ある', 'たまにある', 'ほとんどない', '決してない'], dir_part):
    print(f'  {label}: {val:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
x_u = np.arange(len(ucla_labels_x))
bars = ax.bar(x_u, ucl_part, color=['#c0392b','#e67e22','#3498db','#27ae60'], edgecolor='white', width=0.6)
for bar, val in zip(bars, ucl_part):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.8, f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_xticks(x_u); ax.set_xticklabels(ucla_labels_x, fontsize=9)
ax.set_ylabel('社会活動参加率（%）', fontsize=11)
ax.set_title('UCLA孤独感尺度別 社会参加率', fontsize=12, fontweight='bold')
ax.set_ylim(0, 75); ax.grid(axis='y', alpha=0.3)
# 傾向線
ax.annotate('', xy=(3, ucl_part[3]), xytext=(0, ucl_part[0]),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax = axes[1]
x_d = np.arange(len(direct_labels_x))
bars = ax.bar(x_d, dir_part, color=['#c0392b','#e67e22','#f1c40f','#3498db','#27ae60'], edgecolor='white', width=0.6)
for bar, val in zip(bars, dir_part):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.8, f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_xticks(x_d); ax.set_xticklabels(direct_labels_x, fontsize=9)
ax.set_title('直接質問 孤独感別 社会参加率', fontsize=12, fontweight='bold')
ax.set_ylim(0, 75); ax.grid(axis='y', alpha=0.3)
fig.suptitle('孤独感レベル別 社会活動参加率（令和6年度調査）', fontsize=14, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig10.png')
plt.show()

**【考察 4.2】**
- UCLA孤独感が高いほど社会参加率が低い：10-12点では **24.3%**（全体平均46.6%の約半分）
- 孤独感「決してない（3点）」では **58.2%** と高水準
- 直接質問でも同様の傾向：「しばしばある」31.2% vs「決してない」未計測区間→ 強い正の相関
- **孤独と社会的孤立は相互に強化し合うサイクルを示唆**

## Section 5: 行政・NPO等からの支援状況

In [ ]:
# ── 5.1 行政支援受給状況（都市規模別） ──
df_41 = read_pct(F04, '4-1', n_label_cols=1)
cities_short_11 = ['全体', '政令市・特別区', '20万人以上市', '10-20万人市', '10万人未満市', '町村']
data_41 = df_41.iloc[0:6].fillna(0)
data_41.index = cities_short_11
support_rate = data_41.iloc[:, 0].values

print('=== 行政・NPO等の支援受給率（都市規模別） ===')
for city, val in zip(cities_short_11, support_rate):
    print(f'  {city}: {val:.1f}%')
print(f'\n  注: 全対象者中「支援を受けている」割合（n=8,084）')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors_s = ['#2e86de' if i == 0 else '#54a0ff' for i in range(len(cities_short_11))]
bars = ax.bar(cities_short_11, support_rate, color=colors_s, edgecolor='white')
for bar, val in zip(bars, support_rate):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.15, f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax.axhline(support_rate[0], color='gray', linestyle='--', alpha=0.7, label=f'全体平均 {support_rate[0]:.1f}%')
ax.set_ylabel('支援を受けている割合（%）', fontsize=11)
ax.set_title('行政機関・NPO等からの支援受給状況（都市規模別、令和6年度）', fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10)
ax.set_ylim(0, max(support_rate) * 1.3)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
save_fig(fig, 'fig11.png')
plt.show()

**【考察 5.1】**
- **行政等の支援受給率は全体 7.4%** と低水準 → 約9割以上が何の公的支援も受けていない
- **町村が 10.7%** と最高 → 農村部での互助・行政関与が比較的高い
- **20万人以上の都市で 6.3%** と最低 → 都市部での支援アクセスのミスマッチ
- 「受けていない（85.1%）」「わからない（3.4%）」が大多数 → 支援の認知不足・必要性の不認識が課題

## Section 6: 生活満足度と孤独感の関係

In [ ]:
# ── 6.1 生活満足度別 孤独感分布 ──
df_126 = read_pct(F01, '1-26', n_label_cols=3)
sat_labels = ['全体', '満足している', 'まあ満足している', 'どちらともいえない', 'やや不満である', '不満である']
sat_short_x = ['全体', '満足して
いる', 'まあ満足
している', 'どちらとも
いえない', 'やや
不満', '不満で
ある']
data_126 = df_126.iloc[0:6, 0:4].fillna(0)
data_126.index = sat_labels

print('=== 生活満足度別 UCLA高孤独スコア率 ===')
for idx, val in zip(sat_labels, data_126.iloc[:, 0]):
    print(f'  {idx}: {val:.1f}%')

In [ ]:
data_126.index = sat_short_x
fig, ax = plt.subplots(figsize=(11, 5.5))
colors = ['#c0392b', '#e67e22', '#3498db', '#27ae60']
labels_legend = ['10-12点（常にある）', '7-9点（時々ある）', '4-6点（ほとんどない）', '3点（決してない）']
x = np.arange(len(sat_short_x)); width = 0.65
bottom = np.zeros(len(sat_short_x))
for i, (color, leg) in enumerate(zip(colors, labels_legend)):
    vals = data_126.iloc[:, i].values
    ax.bar(x, vals, width, bottom=bottom, color=color, label=leg, edgecolor='white')
    for j, (v, b) in enumerate(zip(vals, bottom)):
        if v > 3:
            ax.text(x[j], b + v/2, f'{v:.1f}%', ha='center', va='center', fontsize=8.5, color='white', fontweight='bold')
    bottom += vals
ax.set_xticks(x); ax.set_xticklabels(sat_short_x, fontsize=10.5)
ax.set_ylabel('割合（%）', fontsize=11)
ax.set_title('現在の生活満足度別 孤独感分布（UCLA尺度）', fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10, bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
save_fig(fig, 'fig12.png')
plt.show()

**【考察 6.1】**
- **「不満である」層のUCLA高スコア 31.5%**（全体比+25pt）→ 生活不満と孤独感は極めて強く相関
- **「満足している」層はわずか 1.0%** → 生活満足度の高さが孤独感の緩衝として機能
- 「7-9点（時々ある）」も「不満である」層では 44.8% と高水準
- **生活の満足度と孤独感は双方向の影響関係** を示している

## Section 7: 総合考察

### 7.1 主要知見のまとめ

| テーマ | 主要指標 | 全体値 | リスク層 |
|--------|---------|--------|---------|
| **孤独感（UCLA）** | 高スコア（10-12点）率 | 6.5% | 求職中 18.6%、ひとり世帯 9.9%、20-30代 9-10% |
| **孤独感（直接質問）** | 「しばしばある・常にある」率 | 4.3% | 求職中 10.5%、20代 7.4% |
| **社会的交流** | 「全くない（直接会う）」率 | 9.3% | 70代 16.5%、40-50代 12% |
| **社会参加** | 参加率 | 46.6% | 孤独スコア高い群 24.3% |
| **行政支援** | 受給率 | 7.4% | 町村 10.7%（最高）|

### 7.2 孤独感に関連する主要因子（強い順）

```
1. 生活満足度（不満→孤独: +25pt）
2. 就業状況（求職中→孤独: +12pt）
3. 社会参加有無（非参加→孤独: +6pt）
4. 世帯構成（ひとり世帯→孤独: +3.4pt）
5. 年齢（20-30代が高リスク）
6. 都市規模（差は小さい: ±1.6pt）
```

### 7.3 政策的示唆

1. **若年層対策の強化**：20-30代の孤独感は全体最高水準。就労支援・コミュニティ形成の場の提供が急務
2. **単身世帯への支援**：ひとり世帯の孤独感が高い。特に高齢単身者の直接コミュニケーション支援
3. **社会参加の促進**：孤独感の高い層ほど参加率が低く、悪循環に陥りやすい。入口の低い活動の提供が重要
4. **行政支援の認知向上**：受給率わずか7.4%。支援の存在を知らない層への情報提供が必要
5. **就労支援との連携**：求職中の孤独感は特に深刻。ハローワーク等での孤独・孤立への配慮が必要

### 7.4 データの限界

- クロス集計データのみのため因果関係は確認できない
- 調査時点のスナップショットであり、時系列変化は別途年度比較データが必要
- 「無回答」層の特性が不明

In [ ]:
print('分析レポートの全セクション完了')
print(f'生成画像数: 12枚')
print(f'保存先: {IMG_DIR}')